# Appendix: mixtures of multichannel hidden Markov models

Section 4 extends every clustering result to hidden Markov models with multichannel
observations, and is the only part of the paper with no figure. This notebook gives it one,
by running the whole numerical section again on that mechanism and asking whether the
conclusions survive.

**What changes.** A component is now a homogeneous HMM: a latent chain on four hidden
states drives five conditionally independent channels of five letters each. A sequence is an
$(n, 5)$ array rather than an $(n,)$ one, and the dissimilarity is the multichannel OM of
Definition 4.1, $c_{\mathrm{sub}} = \sum_c c_{\mathrm{sub}}^{(c)}$ and
$\delta = \sum_c \delta^{(c)}$, with constant per-channel costs. That aggregation gives
$M^{\mathrm{mc}} = 10$ and $\delta^{\mathrm{mc}} = 5$ against 2 and 1 in the univariate
case, and Assumption 1 holds — checked channel by channel, since the product alphabet has
$5^5 = 3125$ letters and the triangle inequality on it is a 227 GiB table.

The costs are fixed in advance rather than estimated from the data, unlike the TRATE path.
$\Gamma^{(n)}$ is estimated on an independent sample, and a cost scheme derived from that
sample would make the dissimilarity depend on which sequences happened to be drawn.

**What does not change.** Everything downstream of the dissimilarity matrix. The same
estimator of $\Gamma^{(n)}$, the same simultaneous intervals, the same four clustering
algorithms, the same rules for $K$ — none of them knows which mechanism produced the matrix
it reads.

**The grid axis.** The single knob $\alpha$ ties the Dirichlet concentrations of the
initial law, transitions, and emissions. Small values give sharply peaked laws and
components that are easy to tell apart.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clustering import (adjusted_rand_index, asw_select_k, exact_recovery,
                        profile_distances, profile_graph_k, profile_heights,
                        safeguard_threshold as theorem_3_8_threshold)
from experiments import (ResultsWriter, draw_hmm_mixture,
                         estimate_gamma_paths, eta_rows, labels_at_k, multichannel_om,
                         stream, wilson_interval)
from figures import DIVERGING_CMAP, PAPER_STYLE, heatmap, save

## Convergence of $\hat\gamma_n$

The appendix analogue of Section 5.1 on this mechanism, at the $lpha$ the paths
below run at. Self-contained: it draws its own pair of components and does not read
the grid. The bounds of Proposition 2.10 are not drawn -- they are stated on the
stationary law of the product alphabet, ^5 = 3125$ letters here, which this
experiment never forms.

In [ ]:
# --- Convergence of hat-gamma_n, on multichannel HMMs ------------------------
from experiments import MultichannelOM          # noqa: E402
from om import compute_trate_subst_matrix       # noqa: E402

# The appendix analogue of Section 5.1, on this notebook's mechanism. Self-contained
# and independent of the grid below: it draws its own pair of components, so it runs
# on its own in a few minutes.
CONV_ALPHA = 3.0                        # the alpha the appendix paths run at
CONV_STATES, CONV_VARS, CONV_CATS = 4, 5, [5] * 5
CONV_R = 30                             # replicates per configuration
CONV_N_MAX = 10_000                     # the horizon Section 5.1 runs to
CONV_GRID = np.unique(np.round(np.geomspace(10, CONV_N_MAX, 24)).astype(np.int64))
CONV_SEED = 20260901
CONV_FIGURES = Path("Figures/HMM")
COL_IN, COL_OUT = "#1b6ca8", "#c0392b"

conv_mix = draw_hmm_mixture(2, CONV_STATES, CONV_VARS, CONV_CATS, 0, CONV_SEED,
                            alpha=CONV_ALPHA)


def conv_draw(k, tag):
    """CONV_R trajectories of component k, each configuration on its own stream."""
    return conv_mix.sample_component(k, CONV_R, CONV_N_MAX,
                                     stream(CONV_SEED, "hmm-conv", tag))


conv_traj = {"within":  (conv_draw(0, "within-a"),  conv_draw(0, "within-b")),
             "between": (conv_draw(0, "between-a"), conv_draw(1, "between-b"))}

# The pilot the TRATE costs are estimated on, independent of the trajectories above.
conv_pilot = np.concatenate([
    conv_mix.sample_component(k, 2, 2000, stream(CONV_SEED, "hmm-conv", "pilot", k))
    for k in (0, 1)])


def multichannel_trate_om(pilot, n_categories, indel=1.0):
    """Definition 4.1 aggregation, with per-channel TRATE costs read off `pilot`.

    `multichannel_cost_scheme` builds only schemes fixed in advance and refuses TRATE,
    which is data-driven. Estimating it here on a pilot drawn independently of the
    trajectories that enter hat-gamma_n keeps the property that matters -- the
    dissimilarity does not depend on the sequences it scores -- exactly as
    `om_convergence.ipynb` does in the univariate case.
    """
    per_channel = [compute_trate_subst_matrix(pilot[:, :, c], n_states=int(count))
                   for c, count in enumerate(n_categories)]
    packed = np.zeros((len(n_categories),) + (max(n_categories),) * 2)
    for c, S in enumerate(per_channel):
        packed[c, :S.shape[0], :S.shape[0]] = S
    return MultichannelOM(name="trate-mc", costs=packed,
                          indel=float(len(n_categories)) * indel,
                          M_mc=float(sum(S.max() for S in per_channel)),
                          n_categories=tuple(int(c) for c in n_categories),
                          channel_indel=np.full(len(n_categories), float(indel)))


def run_conv_scheme(om_scheme):
    """Assumption 1 on the channels, then the sample paths of both configurations."""
    checks = om_scheme.assumption_1()
    ok = all(v for k, v in checks.items() if k.startswith("("))
    print(f"[{om_scheme.name}] Assumption 1 holds = {ok}, M^mc = {om_scheme.M:.3f}")
    t0 = time.time()
    paths = {k: om_scheme.gamma_paths(A, B, CONV_GRID) for k, (A, B) in conv_traj.items()}
    print(f"[{om_scheme.name}] OM computations: {time.time() - t0:.1f}s"
          f"   within {paths['within'][:, -1].mean():.3f}"
          f"   between {paths['between'][:, -1].mean():.3f}")
    return paths


def plot_conv_hmm(paths, title, filename=None):
    """The layout of `plot_gamma_convergence` in om_convergence.ipynb, without the
    bounds of Proposition 2.8: those are stated on the stationary law of the product
    alphabet -- 3125 letters here -- which this experiment never forms.

    The y axis is not anchored at 0 either. Five channels add their costs, so gamma sits
    near 6.5 while eta stays of order 0.1; anchoring at 0 as the univariate figure does
    would leave the two configurations overlapping in the top tenth of the panel."""
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(5.2, 4.0))
        for key, col, lab in (("within", COL_IN, r"$(P,P)$"),
                              ("between", COL_OUT, r"$(P,Q)$")):
            ax.plot(CONV_GRID, paths[key].T, color=col, lw=0.6, alpha=0.16)
            ax.plot(CONV_GRID, paths[key].mean(0), color=col, lw=1.9,
                    label=rf"$\hat\gamma_n$, {lab}")
        ax.set_xscale("log")
        ax.set_xlabel(r"$n$")
        ax.set_ylabel(r"$\hat\gamma_n = d_{\mathrm{OM}}(X_{1:n},Y_{1:n})\,/\,n$")
        ax.set_title(title, fontsize=10)
        ax.legend(loc="lower right", handlelength=2.6, borderaxespad=0.6)
        fig.tight_layout()
        if filename is not None:
            save(fig, CONV_FIGURES, filename)
        plt.show()
    return fig


for conv_om, conv_title, conv_name in (
    (multichannel_om("constant", CONV_CATS, sub=2.0, indel=1.0),
     r"constant costs: $c_{\mathrm{sub}}\equiv 2$, $\delta\equiv 1$ per channel",
     "gamma_convergence_constant_hmm"),
    (multichannel_trate_om(conv_pilot, CONV_CATS, indel=1.0),
     r"TRATE costs: $c_{\mathrm{sub}}=2-\hat T-\hat T^{T}$ per channel, $\delta\equiv 1$",
     "gamma_convergence_trate_hmm"),
    (multichannel_om("random", CONV_CATS, rng=np.random.default_rng(CONV_SEED + 1),
                     low=1.2, high=2.0, indel=1.0),
     r"random costs: $c_{\mathrm{sub}}\sim\mathcal{U}[1.2,2]$ per channel, $\delta\equiv 1$",
     "gamma_convergence_random_hmm"),
):
    _ = plot_conv_hmm(run_conv_scheme(conv_om), conv_title, conv_name)

## Setup

Everything reported below is computed in this notebook: mixtures, independent estimates
of $\Gamma^{(n)}$, dissimilarity matrices, clustering, rules for $K$, and horizon paths.
As in `difficulty_grid`, every run overwrites its own tables and never reuses rows from
an earlier one, and there is no opt-out that reads a grid computed before: the notebook
reproduces the appendix, it does not store it. The full settings take several hours;
reduce the knobs below for a quick end-to-end run.

In [ ]:
# --- the knobs -------------------------------------------------------------
N = 800                    # sequences per clustering dataset
R_MIX = 30                 # mixtures per (alpha, K) cell
ALPHAS = [1.0, 2.0, 3.0, 5.0, 10.0, 30.0]
KS = list(range(2, 11))
HORIZONS = [1000]
HORIZON = 1000
N_GAMMA = 60
LEVEL = 0.95
SEED = 20260901
PAM_RESTARTS = 1
K_MAX_ASW = 12
HMM_STATES, HMM_VARS, HMM_CATEGORIES = 4, 5, [5] * 5
ALGOS = ["average", "pam"]
RULES = ["threshold", "asw-pam"]
PATH_ALGOS = ["single", "average", "pam"]
PATH_ALPHA, PATH_K = 3.0, 4
PATH_MIXTURES, PATH_DATASETS = 6, 20
PATH_HORIZONS = list(range(50, 551, 50))


RESULTS = Path("results")
FIGURES = Path("Figures/HMM")
FIGURES.mkdir(parents=True, exist_ok=True)
OUT = {kind: RESULTS / f"hmm_notebook_{kind}.csv"
       for kind in ("cluster", "eta", "khat")}
OUT_PATH = RESULTS / "hmm_notebook_path.csv"

FIELDS = {
    "cluster": ["alpha", "K", "d", "mixture_id", "n", "N",
                "cost_scheme", "algorithm", "dataset_id", "exact_recovery",
                "ari", "mixture_key"],
    "eta": ["alpha", "K", "mixture_id", "n", "cost_scheme", "eta_hat",
            "eta_ci_low", "eta_ci_high", "separation_status", "n_pairs",
            "level", "mixture_key"],
    "khat": ["alpha", "K", "mixture_id", "dataset_id", "n", "N", "d",
             "cost_scheme", "rule", "threshold", "k_hat", "k_correct",
             "exact_recovery", "mixture_key"],
}
PATH_FIELDS = FIELDS["cluster"]


om = multichannel_om("constant", HMM_CATEGORIES, sub=2.0, indel=1.0)


def build_mixture(K, alpha, mixture_id):
    return draw_hmm_mixture(K, HMM_STATES, HMM_VARS, HMM_CATEGORIES,
                            mixture_id, SEED, alpha=alpha)


def evaluate_rules(D, rho, heights, n):
    out = []
    threshold = theorem_3_8_threshold(rho, n, heights)
    k_hat, labels = profile_graph_k(None, rho=rho, threshold=threshold,
                                    return_labels=True)
    out.append(("threshold", threshold, k_hat, labels))
    k_hat, labels = asw_select_k(D, k_max=K_MAX_ASW, method="pam",
                                  pam_restarts=PAM_RESTARTS, return_labels=True)
    out.append(("asw-pam", "", k_hat, labels))
    return out


def score_known_k(mixture, matrices, truth, horizons, algorithms, dataset_id, rng):
    """One partition per requested algorithm and horizon, only at known K."""
    rows = []
    for g, n in enumerate(horizons):
        D = matrices[g]
        for algorithm in algorithms:
            labels = labels_at_k(D, mixture.K, algorithm, rng=rng,
                                 pam_restarts=PAM_RESTARTS)
            rows.append({"alpha": mixture.alpha, "K": mixture.K, "d": mixture.d,
                         "mixture_id": mixture.mixture_id, "n": int(n), "N": N,
                         "cost_scheme": om.name, "algorithm": algorithm,
                         "dataset_id": dataset_id,
                         "exact_recovery": int(exact_recovery(labels, truth)),
                         "ari": adjusted_rand_index(labels, truth),
                         "mixture_key": mixture.key})
    return rows


def sweep_unit(alpha, K, m):
    mixture = build_mixture(K, alpha, m)
    estimate = estimate_gamma_paths(
        mixture, om, HORIZONS, N_GAMMA,
        stream(SEED, "hmm", "grid-gamma", alpha, K, m))
    geometry = eta_rows(mixture, om, estimate, level=LEVEL)
    rng = stream(SEED, "hmm", "grid-data", alpha, K, m)
    X, truth = mixture.sample_dataset(N, max(HORIZONS), rng)
    matrices = om.matrices(X, HORIZONS)
    cluster = score_known_k(mixture, matrices, truth, HORIZONS, ALGOS, 0, rng)
    selection = []
    g = HORIZONS.index(HORIZON)
    D = matrices[g]
    rho, heights = profile_distances(D), None
    heights = profile_heights(rho)
    for rule, threshold, k_hat, labels in evaluate_rules(D, rho, heights, HORIZON):
        selection.append({"alpha": alpha, "K": K, "mixture_id": m,
                          "dataset_id": 0, "n": HORIZON, "N": N,
                          "d": mixture.d, "cost_scheme": om.name, "rule": rule,
                          "threshold": threshold, "k_hat": int(k_hat),
                          "k_correct": int(k_hat == K),
                          "exact_recovery": int(exact_recovery(labels, truth)),
                          "mixture_key": mixture.key})
    return geometry, cluster, selection


def compute_grid():
    for path in OUT.values():
        path.unlink(missing_ok=True)
    writers = {kind: ResultsWriter(OUT[kind], FIELDS[kind]) for kind in FIELDS}
    units = [(a, K, m) for a in ALPHAS for K in KS for m in range(R_MIX)]
    start = time.perf_counter()
    try:
        for done, (alpha, K, m) in enumerate(units, 1):
            geometry, cluster, selection = sweep_unit(alpha, K, m)
            for kind, rows in (("eta", geometry), ("cluster", cluster),
                               ("khat", selection)):
                writers[kind].write([{k: row.get(k, "") for k in FIELDS[kind]}
                                     for row in rows])
            if done % 10 == 0 or done == len(units):
                elapsed = time.perf_counter() - start
                left = (len(units) - done) * elapsed / done
                print(f"HMM: {done}/{len(units)}, ~{left/60:.1f} min left",
                      flush=True)
    finally:
        for writer in writers.values():
            writer.close()


def compute_paths():
    OUT_PATH.unlink(missing_ok=True)
    writer = ResultsWriter(OUT_PATH, PATH_FIELDS)
    try:
        for m in range(PATH_MIXTURES):
            mixture = build_mixture(PATH_K, PATH_ALPHA, m)
            for r in range(PATH_DATASETS):
                rng = stream(SEED, "hmm", "path-data", PATH_ALPHA, PATH_K, m, r)
                X, truth = mixture.sample_dataset(N, max(PATH_HORIZONS), rng)
                matrices = om.matrices(X, PATH_HORIZONS)
                rows = score_known_k(mixture, matrices, truth, PATH_HORIZONS,
                                     PATH_ALGOS, r, rng)
                writer.write([{k: row.get(k, "") for k in PATH_FIELDS} for row in rows])
            print(f"HMM paths: mixture {m + 1}/{PATH_MIXTURES}", flush=True)
    finally:
        writer.close()


print(f"computing HMMs: {len(ALPHAS)} x {len(KS)} x {R_MIX} units")
compute_grid()
compute_paths()


def load(kind):
    """The rows of `kind` the sweep above just wrote."""
    return pd.read_csv(OUT[kind])


hmm_cluster, hmm_eta, hmm_khat = (load(kind)
                                      for kind in ("cluster", "eta", "khat"))
hmm_path = pd.read_csv(OUT_PATH)

def at_horizon(df):
    return df[df.n == HORIZON]

hmm_cluster = at_horizon(hmm_cluster)
hmm_eta = at_horizon(hmm_eta)
hmm_khat = at_horizon(hmm_khat)
ALPHAS_PRESENT = ALPHAS
print(f"grids drawn from this run on alpha in {ALPHAS_PRESENT}")

## Reading a cell

In [ ]:
def grid(df, value, agg="mean"):
    table = df.pivot_table(index="alpha", columns="K", values=value, aggfunc=agg)
    return table.reindex(index=ALPHAS_PRESENT, columns=KS).to_numpy(dtype=float)


def verdict_balance(eta):
    """Pr(separated) - Pr(nonseparated): +1 all separated, -1 all not, 0 no verdict."""
    e = eta.assign(sep=(eta.separation_status == "separated").astype(int),
                   non=(eta.separation_status == "nonseparated").astype(int))
    return grid(e, "sep") - grid(e, "non"), grid(e, "sep"), grid(e, "eta_hat", agg="median")

## Plausibility of the separation condition

For the HMM grid, blue where $\eta_n > 0$ is established at 95% simultaneous
confidence, red where $\eta_n < 0$ is, pale where the interval cannot decide.

In [ ]:
balance, _, margin = verdict_balance(hmm_eta)
with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    im = heatmap(ax, balance, ALPHAS_PRESENT, KS, "multichannel HMMs",
                 annot=margin, cmap=DIVERGING_CMAP, vmin=-1.0, vmax=1.0,
                 fmt="{:+.2f}")   # the balance is signed: say so, as the Markov grid does
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, ticks=[-1, 0, 1])
    cb.ax.set_yticklabels(["all not\nseparated", "no\nverdict", "all\nseparated"], fontsize=6)
    cb.outline.set_visible(False)
    fig.tight_layout()
    save(fig, FIGURES, "separation_hmm")
    plt.show()

## Recovery at known $K$

Average linkage and PAM, on the same grid.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)
    for ax, algo in zip(axes, ALGOS):
        sub = hmm_cluster[hmm_cluster.algorithm == algo]
        im = heatmap(ax, grid(sub, "exact_recovery"), ALPHAS_PRESENT, KS, algo,
                     annot=grid(sub, "ari"))
    cb = fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02)
    cb.set_label("probability of exact recovery (annotated: mean ARI)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, FIGURES, "recovery_hmm")
    plt.show()

## Selecting $K$

The threshold of Theorem 3.8 against the applied default, on HMM dissimilarities.

In [ ]:
TITLES = {"threshold": "threshold of Theorem 3.8",
          "asw-pam": "maximal silhouette width (PAM)"}

with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6), sharey=True)
    for ax, rule in zip(axes, RULES):
        sub = hmm_khat[hmm_khat.rule == rule]
        im = heatmap(ax, grid(sub, "k_correct"), ALPHAS_PRESENT, KS, TITLES[rule],
                     annot=grid(sub, "exact_recovery"))
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    cb.set_label(r"$\Pr(\hat K = K)$ (annotated: $\Pr$ exact partition)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, FIGURES, "k_selection_hmm")
    plt.show()

## Recovery against the horizon

In [ ]:
COL = {"single": "#1b6ca8", "average": "#6a51a3", "pam": "#c0392b"}
NAMES = {"single": "single linkage", "average": "average linkage", "pam": "PAM"}
HZ = np.sort(hmm_path.n.unique())


def curves(df, algorithm, value="ari"):
    """One row per dataset path, plus one mean path per mixture."""
    sub = df[df.algorithm == algorithm]
    reps = sub.pivot_table(index=["mixture_id", "dataset_id"], columns="n", values=value)
    means = sub.pivot_table(index="mixture_id", columns="n", values=value)
    return reps.reindex(columns=HZ).to_numpy(), means.reindex(columns=HZ).to_numpy()


def plot_ari_curve(algorithm, filename=None):
    reps, per_mixture = curves(hmm_path, algorithm)
    colour = COL[algorithm]
    with plt.rc_context(PAPER_STYLE):
        fig, ax = plt.subplots(figsize=(5.2, 4.0))
        ax.plot(HZ, reps.T, color=colour, lw=0.6, alpha=0.13)
        ax.plot(HZ, per_mixture.T, color=colour, lw=0.9, alpha=0.5)
        ax.plot(HZ, reps.mean(0), color=colour, lw=1.9,
                label=rf"mean over {reps.shape[0]} replicates")
        ax.plot([], [], color=colour, lw=0.9, alpha=0.5,
                label=rf"{per_mixture.shape[0]} mixture means")
        ax.set_xlabel(r"$n$, length of a sequence")
        ax.set_ylabel(f"ARI, {NAMES[algorithm]}")
        ax.set_ylim(-0.04, 1.04)
        ax.axhline(0.5, color="0.6", lw=0.7, ls=(0, (1, 3)), zorder=0)
        ax.set_title(rf"{NAMES[algorithm]} --- HMM, $N = {int(hmm_path.N.iloc[0])}$, "
                     rf"$K = {PATH_K}$, $\alpha = {PATH_ALPHA}$", fontsize=10)
        ax.legend(loc="lower right", handlelength=2.6, borderaxespad=0.6)
        fig.tight_layout()
        if filename is not None:
            save(fig, FIGURES, filename)
        plt.show()
    return fig


def exact_recovery_table(algorithm):
    """P(ARI = 1) at each horizon, with its Wilson interval."""
    sub = hmm_path[hmm_path.algorithm == algorithm]
    rows = []
    for n in HZ:
        at_n = sub[sub.n == n]
        s = at_n.exact_recovery
        lo, hi = wilson_interval(int(s.sum()), len(s), LEVEL)
        rows.append({"n": int(n), "P(exact)": f"{s.mean():.2f}",
                     "95% CI": f"[{lo:.2f}, {hi:.2f}]",
                     "mean ARI": f"{at_n.ari.mean():.3f}"})
    return pd.DataFrame(rows)


for algorithm in PATH_ALGOS:
    _ = plot_ari_curve(algorithm, filename=f"ari_path_{algorithm}_hmm")
    print(exact_recovery_table(algorithm).to_string(index=False))